# Commodity Term Structure - Research Notebook

**Source**: QC Strategy Library #26 - Term Structure Effect in Commodities

**Concept**: A long-short commodity futures strategy based on roll returns. Calculates annualized roll returns from near vs distant contract price ratios. Top quintile backwardation = long, top quintile contango = short. Monthly rebalance across 21 commodity futures.

In [1]:
from AlgorithmImports import *
qb = QuantBook()

print(f"Universe: 21 commodity futures across 5 sectors")
print(f"  Softs (5): Cocoa, Coffee, Cotton, Orange Juice, Sugar")
print(f"  Grains (6): Corn, Oats, Soybean Meal, Soybean Oil, Soybeans, Wheat")
print(f"  Meats (3): Feeder Cattle, Lean Hogs, Live Cattle")
print(f"  Energies (4): Crude Oil WTI, Heating Oil, Natural Gas, Gasoline")
print(f"  Metals (3): Gold, Palladium, Silver")
print(f"\nRoll return formula: R = (ln(P_near) - ln(P_distant)) * 365 / days_diff")
print(f"Rebalance: Monthly, equal-weighted long-short")

Universe: 21 commodity futures across 5 sectors
  Softs (5): Cocoa, Coffee, Cotton, Orange Juice, Sugar
  Grains (6): Corn, Oats, Soybean Meal, Soybean Oil, Soybeans, Wheat
  Meats (3): Feeder Cattle, Lean Hogs, Live Cattle
  Energies (4): Crude Oil WTI, Heating Oil, Natural Gas, Gasoline
  Metals (3): Gold, Palladium, Silver

Roll return formula: R = (ln(P_near) - ln(P_distant)) * 365 / days_diff
Rebalance: Monthly, equal-weighted long-short


## Strategy Logic

### Term Structure Effect
1. **Roll Return**: Annualized log-price difference between near and distant futures contracts
2. **Backwardation** (positive roll return): near contract trades at premium -> long signal
3. **Contango** (negative roll return): distant contract trades at premium -> short signal
4. **Selection**: Top quintile backwardation = long, top quintile contango = short
5. **Rebalance**: Monthly, equal-weighted within each side

### Economic Rationale
- **Theory of Normal Backwardation** (Keynes 1930): hedgers pay risk premium to speculators
- Commodity producers hedge downside (sell futures), creating upward pressure on near contracts
- Strong backwardation suggests supply tightness -> upward price pressure
- Strong contango suggests oversupply -> downward price pressure

### Roll Return Calculation
```
R = (ln(P_near) - ln(P_distant)) * 365 / (T_distant - T_near)
```
- Near contract: earliest expiry in the chain
- Distant contract: latest expiry in the chain (up to 90 days)
- Annualized by dividing by the expiry difference in days

In [2]:
import numpy as np
np.random.seed(42)
n_commodities = 21
roll_returns = np.random.normal(0, 0.15, n_commodities)
print("Simulated Roll Returns (21 commodities):")
print(f"  Mean: {roll_returns.mean():.4f}")
print(f"  Std: {roll_returns.std():.4f}")
print(f"  Positive (backwardation): {(roll_returns > 0).sum()}")
print(f"  Negative (contango): {(roll_returns < 0).sum()}")
quintile = n_commodities // 5
print(f"\nQuintile size: {quintile}")
print(f"Long positions (top backwardation): {quintile}")
print(f"Short positions (top contango): {quintile}")
print(f"Total positions: {quintile * 2}")
print(f"Weight per position: {0.1 / quintile:.1%}")

Simulated Roll Returns (21 commodities):
  Mean: -0.0140
  Std: 0.1466
  Positive (backwardation): 9
  Negative (contango): 12

Quintile size: 4
Long positions (top backwardation): 4
Short positions (top contango): 4
Total positions: 8
Weight per position: 2.5%


## Key Observations

- **Roll returns** capture the term structure premium, distinct from spot price changes
- **Long-short structure** provides partial hedging against broad commodity moves
- **Quintile ranking** is robust to absolute roll return levels (relative, not absolute)
- **Monthly rebalance** aligns with futures roll cycles

### Strengths
- Grounded in economic theory (Keynes 1930, theory of normal backwardation)
- 21 commodities across 5 sectors provide diversification
- Long-short structure reduces exposure to broad commodity trends
- Simple, transparent signal (roll return ranking)
- No ML or complex parameters - low overfitting risk

### Risks
- Long-term performance is negative (-15.71% CAGR over 5Y), suggesting regime dependency
- 80% drawdown indicates extreme tail risk during certain periods
- Recent strong performance (38.85% 3M) may not persist
- 21 commodities may not provide enough cross-sectional variation
- Futures contracts have roll risk near expiry
- Commodity markets can be dominated by a few large hedgers, distorting term structure

In [3]:
print("=" * 50)
print("BACKTEST RESULTS (Author Reported)")
print("=" * 50)
print("Long-term (5Y OOS):")
print("  CAGR: -15.71%")
print("  Max Drawdown: 80.80%")
print("  Sharpe: -0.041")
print()
print("Recent (OOS):")
print("  3M Return: 38.85%")
print("  1Y Sharpe: 1.49")
print("  3M Sharpe: 10.52")
print()
print("Run backtest via QC MCP to verify:")
print("  create_compile -> create_backtest -> read_backtest")
print()
print("Characteristics:")
print("  Universe: 21 commodity futures (5 sectors)")
print("  Signal: Annualized roll return ranking")
print("  Selection: Top quintile backwardation (long) + contango (short)")
print("  Rebalance: Monthly, equal-weighted")
print("  Exposure: 10% gross per side")

BACKTEST RESULTS (Author Reported)
Long-term (5Y OOS):
  CAGR: -15.71%
  Max Drawdown: 80.80%
  Sharpe: -0.041

Recent (OOS):
  3M Return: 38.85%
  1Y Sharpe: 1.49
  3M Sharpe: 10.52

Run backtest via QC MCP to verify:
  create_compile -> create_backtest -> read_backtest

Characteristics:
  Universe: 21 commodity futures (5 sectors)
  Signal: Annualized roll return ranking
  Selection: Top quintile backwardation (long) + contango (short)
  Rebalance: Monthly, equal-weighted
  Exposure: 10% gross per side
